# 04e — How long a run takes, judging included

The run record carries the conversation's time (`study_case_executions`
`started_at` / `finished_at`) and the run's wall clock (`study_runs`), but
**judging is a separate pass** after the run closes (`tools/overnight_baseline.py`),
so its time is read from the `created_at` of the rows it writes: one
`study_scores` row per point-wise call, one `study_comparisons` row per pair-wise
comparison (both orderings). Each pass is sequential, so the time between
consecutive rows is the time one execution's judging took. Same two runs as
04b–04d: the baseline (`1a872c09`, D96) and the regression run (`e097f78e`).

**Idle time is not work.** A gap between consecutive rows longer than `PAUSE_S`
(five minutes) is a pause — the baseline was left overnight, the pair-wise pass
stopped once before a resume — and is excluded. Every excluded gap is listed in
section 1, so nothing is dropped silently.

**Overhead** is the execution phase's active time that the conversations do not
account for: mostly building the next repetition's fixture trip.

**Every run was sequential, so the parallel figures in section 4 are
extrapolations, not measurements.** They compose serial durations and assume a
parallel run is as fast per execution as a serial one. It would not be: the
harness's rate limiter (`VERTEX_CALLS_PER_MINUTE`, ten a minute, shared by
agent, simulator and judge) would throttle concurrent executions unless raised,
and latency under contention is a different quantity (`runner.py`, D39).

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[1]))

import pandas as pd

from src import runtime_estimate as rt
from src.cases import load_cases

pd.set_option("display.width", 200)

# --- parameters -------------------------------------------------------------
RUNS = {
    "baseline": "1a872c09-7070-4789-a806-904969f9185d",    # B10, adopted (D96)
    "regression": "e097f78e-3326-4aea-acaa-0b904d3b0a1c",  # B11
}
PAUSE_S = rt.PAUSE_S
# ---------------------------------------------------------------------------

runs = {name: rt.load_run(rid) for name, rid in RUNS.items()}
print(f"suite: {len(load_cases())} cases")
execs = {name: rt.per_execution(r, PAUSE_S) for name, r in runs.items()}
for name, r in runs.items():
    wall = (r["run"]["finished_at"].iloc[0] - r["run"]["started_at"].iloc[0]).total_seconds() / 60
    print(f"{name}: {len(r['executions'])} executions, {len(r['scores'])} point-wise scores, "
          f"{len(r['comparisons'])} comparisons; run row wall clock {wall:.1f} min")

suite: 31 cases
baseline: 176 executions, 176 point-wise scores, 0 comparisons; run row wall clock 597.1 min
regression: 176 executions, 176 point-wise scores, 176 comparisons; run row wall clock 106.2 min


## 1. Phases

Per run: the execution phase and each judge pass, with the pauses excluded from
the active time.

In [2]:
ph = pd.concat({name: rt.phases(r, PAUSE_S) for name, r in runs.items()}, names=["run"])
ph.round(1)

rows  span min  pauses  paused min  active min  per execution s
run        phase                                                                            
baseline   execution          176     596.9       1       489.3       107.6             36.7
           point-wise judge   176      17.6       0         0.0        17.7              6.0
regression execution          176     106.0       0         0.0       106.0             36.1
           point-wise judge   176      17.5       0         0.0        17.6              6.0
           pair-wise judge    176      43.6       1         8.6        35.2             12.0

## 2. Per execution

One execution is one repetition of one case. The conversation is measured per
execution; overhead and judging are the run's mean per execution. The baseline
run has no pair-wise pass.

In [3]:
pd.concat({name: rt.summary(f) for name, f in execs.items()}, names=["run"])

conversation s  overhead s  point-wise s  pair-wise s  total s
run        provenance                                                                 
baseline   metamorphic            22.4        10.9           6.0          0.0     39.4
           seeded                 28.1        10.9           6.0          0.0     45.0
regression metamorphic            22.4        11.1           6.0         12.0     51.4
           seeded                 26.9        11.1           6.0         12.0     56.0

## 3. Per test case

Two ways of running one case's repetitions (eight for a seed, four for a
metamorphic case): **sequential**, as the runs did, and **parallel**, where the
case takes as long as its slowest repetition. Minutes, summarised over the cases
of each provenance.

In [4]:
def case_summary(f):
    c = rt.per_case(f)
    out = c.groupby("provenance")[["sequential s", "parallel s"]].agg(["median", "max"]) / 60
    out.columns = [f"{a.split()[0]} min, {b}" for a, b in out.columns]
    return out.round(2)

pd.concat({name: case_summary(f) for name, f in execs.items()}, names=["run"])

sequential min, median  sequential min, max  parallel min, median  parallel min, max
run        provenance                                                                                       
baseline   metamorphic                    2.41                 4.39                  0.68               1.14
           seeded                         4.96                 8.92                  0.79               1.18
regression metamorphic                    3.21                 5.21                  0.84               1.35
           seeded                         6.45                10.85                  0.88               1.63

## 4. The whole suite under four schedules

Only the first row is measured; the other three are composed from it.

In [5]:
sched = pd.DataFrame({name: rt.schedules(f) / 60 for name, f in execs.items()}).round(1)
sched.columns = [f"{c} (min)" for c in sched.columns]
sched

,baseline (min),regression (min)
everything in sequence (measured),125.3,158.8
"cases in parallel, repetitions in sequence",8.9,10.9
"cases in sequence, repetitions in parallel",24.5,29.9
everything in parallel,1.2,1.6


## 5. For the report

The figures as prose rounds them, computed rather than typed.

In [6]:
r = execs["regression"]
cases = rt.per_case(r)
seed, meta = (cases[cases["provenance"] == p] for p in ("seeded", "metamorphic"))
s = rt.schedules(r)
ph_r = ph.loc["regression"]
print(f"Executions were run in sequence. One execution (building the fixture trip, the conversation "
      f"and both judge calls) took {rt.say(r['total s'].mean())} on average, of which the conversation "
      f"was {rt.say(r['conversation s'].mean())}. A seed case's eight repetitions took "
      f"{rt.say(seed['sequential s'].median())} and a metamorphic case's four "
      f"{rt.say(meta['sequential s'].median())} (medians). The regression run's executions took "
      f"{rt.say(ph_r.loc['execution', 'active min'] * 60)}, and with both judge passes "
      f"{rt.say(ph_r['active min'].sum() * 60)}. Run in parallel, a case would take as long as its "
      f"slowest repetition, {rt.say(seed['parallel s'].median())} for a typical case, and the whole "
      f"suite {rt.say(s['everything in parallel'])}; neither configuration was measured.")

Executions were run in sequence. One execution (building the fixture trip, the conversation and both judge calls) took about 55 seconds on average, of which the conversation was about 25 seconds. A seed case's eight repetitions took about 6.5 minutes and a metamorphic case's four about 3 minutes (medians). The regression run's executions took about 1 h 50 min, and with both judge passes about 2 h 40 min. Run in parallel, a case would take as long as its slowest repetition, about 55 seconds for a typical case, and the whole suite about 1.5 minutes; neither configuration was measured.